In [ ]:
# Libraries.
import pandas as pd
import numpy as np
from scipy.spatial.distance import correlation
from scipy.stats import pearsonr

In [64]:
# Read in data.
gtexEP = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/gtexExpressionProfile.parquet")
emtabEP = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/emtabExpressionProfile.parquet")
orthologTable = pd.read_csv("/Users/andrewhsu/Projects/McNair/data/orthologTable.txt", sep="\t")
orthologTableIDs = orthologTable[~orthologTable["Mouse gene stable ID"].isna()].loc[:, ["Gene stable ID", "Mouse gene stable ID"]]

In [13]:
orthologTable[orthologTable["Gene stable ID"].isin(gtexEP[gtexEP.iloc[:, :-1].sum(axis=1) == 0].index)]

,Gene stable ID,Mouse gene stable ID,Mouse homology type,Mouse Gene-order conservation score
111,ENSG00000292338,NaN,NaN,NaN
113,ENSG00000292354,NaN,NaN,NaN
114,ENSG00000292363,NaN,NaN,NaN
115,ENSG00000292336,NaN,NaN,NaN
118,ENSG00000292345,NaN,NaN,NaN
...,...,...,...,...
10080,ENSG00000286053,ENSMUSG00000117809,ortholog_one2one,50.0
10152,ENSG00000237671,NaN,NaN,NaN
16553,ENSG00000286140,ENSMUSG00000117748,ortholog_one2one,100.0
18012,ENSG00000284631,NaN,NaN,NaN


In [14]:
gtexEP

,Brain,Colon,Esophagus,Heart,Kidney,Liver,Pancreas,Stomach,Gene type
Gene stable ID,,,,,,,,,
ENSG00000000003,5.799691,22.906008,18.107586,3.414238,15.745596,23.313591,7.128223,10.594314,protein_coding
ENSG00000000005,0.169146,0.753748,0.284215,0.258039,0.956031,0.018972,0.050229,0.206583,protein_coding
ENSG00000000419,21.387617,40.514694,43.498722,24.609381,25.236029,22.603436,22.147223,37.476429,protein_coding
ENSG00000000457,2.830432,6.585192,6.015115,1.902403,3.748399,4.093346,3.044870,5.295540,protein_coding
ENSG00000000460,1.347589,2.231597,2.198975,0.689768,0.988887,1.240150,0.699461,1.596039,protein_coding
...,...,...,...,...,...,...,...,...,...
ENSG00000310553,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA
ENSG00000310554,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA
ENSG00000310555,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA


In [15]:
emtabEP

,Brain,Colon,Esophagus,Heart,Kidney,Liver,Pancreas,Stomach,Gene type
Gene stable ID,,,,,,,,,
ENSMUSG00000000001,34.911248,129.180584,143.270691,72.898048,72.920467,45.744554,8.901608,54.003405,protein_coding
ENSMUSG00000000003,0.000000,0.000000,0.000000,0.053683,0.000000,0.000000,0.000000,0.000000,protein_coding
ENSMUSG00000000028,2.236124,11.323525,5.613315,22.680435,2.280480,0.871723,0.181636,3.491172,protein_coding
ENSMUSG00000000031,2.567429,1.137826,2650.257043,17.372786,0.758899,1.307234,1.672216,2.287438,lncRNA
ENSMUSG00000000037,1.226929,2.675817,3.285196,0.774847,0.333726,0.011223,0.000000,0.287549,protein_coding
...,...,...,...,...,...,...,...,...,...
ENSMUSG00000109574,0.332800,0.047076,0.021773,0.015338,0.008382,0.000000,0.000000,0.000000,TEC
ENSMUSG00000109575,1.098742,0.000000,0.000000,0.009073,0.000000,0.000000,0.000000,0.000000,TEC
ENSMUSG00000109576,0.022506,0.000000,0.000000,0.024401,0.012853,0.000000,0.000000,0.000000,TEC


In [16]:
orthologTable

,Gene stable ID,Mouse gene stable ID,Mouse homology type,Mouse Gene-order conservation score
0,ENSG00000198888,ENSMUSG00000064341,ortholog_one2one,50.0
1,ENSG00000198763,ENSMUSG00000064345,ortholog_one2one,75.0
2,ENSG00000198804,ENSMUSG00000064351,ortholog_one2one,100.0
3,ENSG00000198712,ENSMUSG00000064354,ortholog_one2one,100.0
4,ENSG00000228253,ENSMUSG00000064356,ortholog_one2one,100.0
...,...,...,...,...
28009,ENSG00000081692,ENSMUSG00000036819,ortholog_one2one,75.0
28010,ENSG00000157873,ENSMUSG00000022074,ortholog_one2many,0.0
28011,ENSG00000157873,ENSMUSG00000042333,ortholog_one2many,100.0
28012,ENSG00000132676,ENSMUSG00000068921,ortholog_one2one,75.0


In [65]:
orthologTest = tuple(orthologTableIDs.iloc[0, :])
humanOrtholog = gtexEP[gtexEP.index.isin(orthologTest)]
mouseOrtholog = emtabEP[emtabEP.index.isin(orthologTest)]

In [18]:
# Euclidean distance. My own methodology.
def euclideanDist(humanOrtholog, mouseOrtholog):
    distSum = 0
    for i in range(0, humanOrtholog.shape[1] - 1):
        distSum += np.square(humanOrtholog.iloc[0, i] - mouseOrtholog.iloc[0, i])
    return np.sqrt(distSum)

In [19]:
euclideanDist(humanOrtholog, mouseOrtholog)

np.float64(44492.547301776765)

In [20]:
np.linalg.norm(humanOrtholog.iloc[0, :-1] - mouseOrtholog.iloc[0, :-1])

np.float64(44492.547301776765)

In [66]:
# Pearson Distance. My own methodology.
def pearsonDist(humanOrtholog, mouseOrtholog):
    # Calculates the Z-Scores for our vectors, and uses these Z-Score vectors to calculate Pearson Distance. The formula was provided in Piasecka et al. 2012.
    x = humanOrtholog.iloc[0, :-1].values.astype(float)
    y = mouseOrtholog.iloc[0, :-1].values.astype(float)
    return np.nan if x.min() == x.max() or y.min() == y.max() else correlation(x, y)

In [67]:
pearsonDist(humanOrtholog, mouseOrtholog)

np.float64(0.20943055191829996)

In [68]:
# Pearson Distance is 1 - r, and I found online that r is just the correlation matrix between two dataframes. So I tried that method.
corr = humanOrtholog.iloc[:, :-1].reset_index(drop=True).corrwith(mouseOrtholog.iloc[:, :-1].reset_index(drop=True), method="pearson", axis=1)
1 - corr[0]

np.float64(0.20943055191829985)

In [74]:
np.array(humanOrtholog.iloc[:, :-1])

array([[37976.71 , 21659.441, 16631.998, 37826.426, 32570.627, 22304.053,
         8863.886, 26384.412]], dtype=float32)

In [73]:
1 - pearsonr(np.array(humanOrtholog.iloc[0, :-1]), np.array(mouseOrtholog.iloc[0, :-1]))

AttributeError: 'numpy.dtypes.ObjectDType' object has no attribute 'dtype'

In [27]:
x = humanOrtholog.iloc[0, :-1].values.astype(float)
y = mouseOrtholog.iloc[0, :-1].values.astype(float)
correlation(x, y)

np.float64(0.20943055191829996)

In [28]:
# TEC
def TEC(humanOrtholog, mouseOrtholog):
    # Turns the vectors binary. So if the expression is greater than 1, we consider that "expressed."
    humanOrthoBinary = (humanOrtholog.iloc[:, :-1] > 1).iloc[0, :]
    mouseOrthoBinary = (mouseOrtholog.iloc[:, :-1] > 1).iloc[0, :]

    humanOnlyTissueNum = (humanOrthoBinary & ~mouseOrthoBinary).sum()
    mouseOnlyTissueNum = (mouseOrthoBinary & ~humanOrthoBinary).sum()

    humanTotalTissue = humanOrthoBinary.sum()
    mouseTotalTissue = mouseOrthoBinary.sum()

    if humanTotalTissue == 0 and mouseTotalTissue == 0:
        return np.nan
    elif humanTotalTissue == 0 or mouseTotalTissue == 0:
        return np.nan
    else:
        return ((humanOnlyTissueNum / humanTotalTissue) + (mouseOnlyTissueNum / mouseTotalTissue)) / 2


In [29]:
# This cell just gets the distance and TEC values, so I can append them to the expression profiles later.
myEuclideanDistArr = []
myPearsonDistArr = []
myTECArr = []
for i in range(0, orthologTableIDs.shape[0]):
    orthologTest = tuple(orthologTableIDs.iloc[i, :])
    humanOrtholog = gtexEP[gtexEP.index.isin(orthologTest)]
    mouseOrtholog = emtabEP[emtabEP.index.isin(orthologTest)]

    if not humanOrtholog.empty and not mouseOrtholog.empty:
        myEuclideanDistArr.append((orthologTest[0], orthologTest[1], np.linalg.norm(humanOrtholog.iloc[0, :-1] - mouseOrtholog.iloc[0, :-1])))
        myPearsonDistArr.append((orthologTest[0], orthologTest[1], pearsonDist(humanOrtholog, mouseOrtholog)))
        myTECArr.append((orthologTest[0], orthologTest[1], TEC(humanOrtholog, mouseOrtholog)))

In [30]:
myEuclidDistDF = pd.DataFrame(myEuclideanDistArr, columns=["Human ID", "Mouse ID", "EuclidDist"])
myPearDistDF = pd.DataFrame(myPearsonDistArr, columns=["Human ID", "Mouse ID", "PearDist"])
myTECDF = pd.DataFrame(myTECArr, columns=["Human ID", "Mouse ID", "TEC"])

In [31]:
# The next 4 cells merge the expression profile dataframe with each distance and TEC column.
orthologTableEuclid = orthologTable.merge(myEuclidDistDF, left_on=["Gene stable ID", "Mouse gene stable ID"], right_on=["Human ID", "Mouse ID"], how="left").drop(columns=["Human ID", "Mouse ID"])
orthologTableEuclidPear = orthologTableEuclid.merge(myPearDistDF, left_on=["Gene stable ID", "Mouse gene stable ID"], right_on=["Human ID", "Mouse ID"], how="left").drop(columns=["Human ID", "Mouse ID"])
orthologTableEuclidPearTEC = orthologTableEuclidPear.merge(myTECDF, left_on=["Gene stable ID", "Mouse gene stable ID"], right_on=["Human ID", "Mouse ID"], how="left").drop(columns=["Human ID", "Mouse ID"])
orthologTableEuclidPearTEC.to_csv("/Users/andrewhsu/Projects/McNair/data/orthologTableDist.csv", index=False)
orthologTableEuclidPearTEC.to_parquet("/Users/andrewhsu/Projects/McNair/data/orthologTableDist.parquet", index=False)

In [32]:
orthologTableEuclidPearTEC

,Gene stable ID,Mouse gene stable ID,Mouse homology type,Mouse Gene-order conservation score,EuclidDist,PearDist,TEC
0,ENSG00000198888,ENSMUSG00000064341,ortholog_one2one,50.0,44492.547302,0.209431,0.0000
1,ENSG00000198763,ENSMUSG00000064345,ortholog_one2one,75.0,52572.771576,0.297247,0.0000
2,ENSG00000198804,ENSMUSG00000064351,ortholog_one2one,100.0,82619.259132,0.235434,0.0000
3,ENSG00000198712,ENSMUSG00000064354,ortholog_one2one,100.0,146743.617287,NaN,NaN
4,ENSG00000228253,ENSMUSG00000064356,ortholog_one2one,100.0,72933.323800,NaN,NaN
...,...,...,...,...,...,...,...
28009,ENSG00000081692,ENSMUSG00000036819,ortholog_one2one,75.0,16.599178,0.996874,0.0625
28010,ENSG00000157873,ENSMUSG00000022074,ortholog_one2many,0.0,194.970650,0.384453,0.2500
28011,ENSG00000157873,ENSMUSG00000042333,ortholog_one2many,100.0,178.921829,0.688115,0.0625
28012,ENSG00000132676,ENSMUSG00000068921,ortholog_one2one,75.0,243.094613,0.922319,0.0000


In [33]:
gtexEP

,Brain,Colon,Esophagus,Heart,Kidney,Liver,Pancreas,Stomach,Gene type
Gene stable ID,,,,,,,,,
ENSG00000000003,5.799691,22.906008,18.107586,3.414238,15.745596,23.313591,7.128223,10.594314,protein_coding
ENSG00000000005,0.169146,0.753748,0.284215,0.258039,0.956031,0.018972,0.050229,0.206583,protein_coding
ENSG00000000419,21.387617,40.514694,43.498722,24.609381,25.236029,22.603436,22.147223,37.476429,protein_coding
ENSG00000000457,2.830432,6.585192,6.015115,1.902403,3.748399,4.093346,3.044870,5.295540,protein_coding
ENSG00000000460,1.347589,2.231597,2.198975,0.689768,0.988887,1.240150,0.699461,1.596039,protein_coding
...,...,...,...,...,...,...,...,...,...
ENSG00000310553,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA
ENSG00000310554,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA
ENSG00000310555,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA


In [34]:
gtexEPNormalized = gtexEP.iloc[:, :-1].div(np.linalg.norm(gtexEP.iloc[:, :-1], axis=1), axis=0)
gtexEPNormalized["Gene type"] = gtexEP["Gene type"]
gtexEPNormalized

,Brain,Colon,Esophagus,Heart,Kidney,Liver,Pancreas,Stomach,Gene type
Gene stable ID,,,,,,,,,
ENSG00000000003,0.134754,0.532214,0.420724,0.079329,0.365844,0.541684,0.165622,0.246156,protein_coding
ENSG00000000005,0.129589,0.577476,0.217749,0.197694,0.732454,0.014535,0.038482,0.158272,protein_coding
ENSG00000000419,0.244700,0.463536,0.497677,0.281560,0.288730,0.258610,0.253390,0.428775,protein_coding
ENSG00000000457,0.224259,0.521753,0.476585,0.150730,0.296990,0.324321,0.241249,0.419572,protein_coding
ENSG00000000460,0.320675,0.531036,0.523273,0.164138,0.235317,0.295109,0.166445,0.379797,protein_coding
...,...,...,...,...,...,...,...,...,...
ENSG00000310553,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,lncRNA
ENSG00000310554,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,lncRNA
ENSG00000310555,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,lncRNA


In [35]:
emtabEPNormalized = emtabEP.iloc[:, :-1].div(np.linalg.norm(emtabEP.iloc[:, :-1], axis=1), axis=0)
emtabEPNormalized["Gene type"] = emtabEP["Gene type"]
emtabEPNormalized

,Brain,Colon,Esophagus,Heart,Kidney,Liver,Pancreas,Stomach,Gene type
Gene stable ID,,,,,,,,,
ENSMUSG00000000001,0.150022,0.555121,0.615670,0.313261,0.313357,0.196576,0.038252,0.232066,protein_coding
ENSMUSG00000000003,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,protein_coding
ENSMUSG00000000028,0.084680,0.428812,0.212571,0.858888,0.086360,0.033011,0.006878,0.132208,protein_coding
ENSMUSG00000000031,0.000969,0.000429,0.999977,0.006555,0.000286,0.000493,0.000631,0.000863,lncRNA
ENSMUSG00000000037,0.272635,0.594590,0.730000,0.172178,0.074157,0.002494,0.000000,0.063896,protein_coding
...,...,...,...,...,...,...,...,...,...
ENSMUSG00000109574,0.986744,0.139579,0.064558,0.045477,0.024854,0.000000,0.000000,0.000000,TEC
ENSMUSG00000109575,0.999966,0.000000,0.000000,0.008258,0.000000,0.000000,0.000000,0.000000,TEC
ENSMUSG00000109576,0.632243,0.000000,0.000000,0.685489,0.361072,0.000000,0.000000,0.000000,TEC


In [36]:
myEuclideanDistNormArr = []
myPearsonDistNormArr = []
for i in range(0, orthologTableIDs.shape[0]):
    orthologTest = tuple(orthologTableIDs.iloc[i, :])
    humanOrtholog = gtexEPNormalized[gtexEPNormalized.index.isin(orthologTest)]
    mouseOrtholog = emtabEPNormalized[emtabEPNormalized.index.isin(orthologTest)]

    if not humanOrtholog.empty and not mouseOrtholog.empty:
        myEuclideanDistNormArr.append((orthologTest[0], orthologTest[1], np.linalg.norm(humanOrtholog.iloc[0, :-1] - mouseOrtholog.iloc[0, :-1])))

myEuclidDistNormDF = pd.DataFrame(myEuclideanDistNormArr, columns=["Human ID", "Mouse ID", "EuclidDistNorm"])

orthologTableEuclidPearTECNorm = orthologTableEuclidPearTEC.merge(myEuclidDistNormDF, left_on=["Gene stable ID", "Mouse gene stable ID"], right_on=["Human ID", "Mouse ID"], how="left").drop(columns=["Human ID", "Mouse ID"])

In [37]:
orthologTableEuclidPearTECNorm

,Gene stable ID,Mouse gene stable ID,Mouse homology type,Mouse Gene-order conservation score,EuclidDist,PearDist,TEC,EuclidDistNorm
0,ENSG00000198888,ENSMUSG00000064341,ortholog_one2one,50.0,44492.547302,0.209431,0.0000,0.435805
1,ENSG00000198763,ENSMUSG00000064345,ortholog_one2one,75.0,52572.771576,0.297247,0.0000,0.509245
2,ENSG00000198804,ENSMUSG00000064351,ortholog_one2one,100.0,82619.259132,0.235434,0.0000,0.416690
3,ENSG00000198712,ENSMUSG00000064354,ortholog_one2one,100.0,146743.617287,NaN,NaN,NaN
4,ENSG00000228253,ENSMUSG00000064356,ortholog_one2one,100.0,72933.323800,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
28009,ENSG00000081692,ENSMUSG00000036819,ortholog_one2one,75.0,16.599178,0.996874,0.0625,0.668685
28010,ENSG00000157873,ENSMUSG00000022074,ortholog_one2many,0.0,194.970650,0.384453,0.2500,0.543398
28011,ENSG00000157873,ENSMUSG00000042333,ortholog_one2many,100.0,178.921829,0.688115,0.0625,0.834154
28012,ENSG00000132676,ENSMUSG00000068921,ortholog_one2one,75.0,243.094613,0.922319,0.0000,0.516088


In [38]:
newColOrder = list(orthologTableEuclidPearTECNorm.columns[:5]) + list(orthologTableEuclidPearTECNorm.columns[-1:]) + list(orthologTableEuclidPearTECNorm.columns[5:7])
orthologTableEuclidPearTECNorm = orthologTableEuclidPearTECNorm.loc[:, newColOrder]

In [39]:
orthologTableEuclidPearTECNorm.to_csv("/Users/andrewhsu/Projects/McNair/data/orthologTableDist.csv", index=False)
orthologTableEuclidPearTECNorm.to_parquet("/Users/andrewhsu/Projects/McNair/data/orthologTableDist.parquet", index=False)

In [40]:
orthologTableEuclidPearTECNorm

,Gene stable ID,Mouse gene stable ID,Mouse homology type,Mouse Gene-order conservation score,EuclidDist,EuclidDistNorm,PearDist,TEC
0,ENSG00000198888,ENSMUSG00000064341,ortholog_one2one,50.0,44492.547302,0.435805,0.209431,0.0000
1,ENSG00000198763,ENSMUSG00000064345,ortholog_one2one,75.0,52572.771576,0.509245,0.297247,0.0000
2,ENSG00000198804,ENSMUSG00000064351,ortholog_one2one,100.0,82619.259132,0.416690,0.235434,0.0000
3,ENSG00000198712,ENSMUSG00000064354,ortholog_one2one,100.0,146743.617287,NaN,NaN,NaN
4,ENSG00000228253,ENSMUSG00000064356,ortholog_one2one,100.0,72933.323800,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
28009,ENSG00000081692,ENSMUSG00000036819,ortholog_one2one,75.0,16.599178,0.668685,0.996874,0.0625
28010,ENSG00000157873,ENSMUSG00000022074,ortholog_one2many,0.0,194.970650,0.543398,0.384453,0.2500
28011,ENSG00000157873,ENSMUSG00000042333,ortholog_one2many,100.0,178.921829,0.834154,0.688115,0.0625
28012,ENSG00000132676,ENSMUSG00000068921,ortholog_one2one,75.0,243.094613,0.516088,0.922319,0.0000


In [41]:
gtexEPLog = np.log2(gtexEP.iloc[:, :-1] + 1)
emtabEPLog = np.log2(emtabEP.iloc[:, :-1] + 1)

In [42]:
gtexEP

,Brain,Colon,Esophagus,Heart,Kidney,Liver,Pancreas,Stomach,Gene type
Gene stable ID,,,,,,,,,
ENSG00000000003,5.799691,22.906008,18.107586,3.414238,15.745596,23.313591,7.128223,10.594314,protein_coding
ENSG00000000005,0.169146,0.753748,0.284215,0.258039,0.956031,0.018972,0.050229,0.206583,protein_coding
ENSG00000000419,21.387617,40.514694,43.498722,24.609381,25.236029,22.603436,22.147223,37.476429,protein_coding
ENSG00000000457,2.830432,6.585192,6.015115,1.902403,3.748399,4.093346,3.044870,5.295540,protein_coding
ENSG00000000460,1.347589,2.231597,2.198975,0.689768,0.988887,1.240150,0.699461,1.596039,protein_coding
...,...,...,...,...,...,...,...,...,...
ENSG00000310553,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA
ENSG00000310554,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA
ENSG00000310555,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA


In [43]:
gtexEPLog

,Brain,Colon,Esophagus,Heart,Kidney,Liver,Pancreas,Stomach
Gene stable ID,,,,,,,,
ENSG00000000003,2.765469,4.579301,4.256073,2.142164,4.065710,4.603691,3.022940,3.535346
ENSG00000000005,0.225455,0.810441,0.360887,0.331176,0.967929,0.027115,0.070704,0.270927
ENSG00000000419,4.484629,5.375550,5.475692,4.678600,4.713478,4.560925,4.532767,5.265903
ENSG00000000457,1.937507,2.923186,2.810467,1.537248,2.247441,2.348614,2.016093,2.654330
ENSG00000000460,1.231180,1.692248,1.677610,0.756825,0.991961,1.163596,0.765077,1.376312
...,...,...,...,...,...,...,...,...
ENSG00000310553,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
ENSG00000310554,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
ENSG00000310555,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [44]:
emtabEPLog

,Brain,Colon,Esophagus,Heart,Kidney,Liver,Pancreas,Stomach
Gene stable ID,,,,,,,,
ENSMUSG00000000001,5.166364,7.024370,7.172634,6.207464,6.207902,5.546726,3.307663,5.781449
ENSMUSG00000000003,0.000000,0.000000,0.000000,0.075441,0.000000,0.000000,0.000000,0.000000
ENSMUSG00000000028,1.694267,3.623343,2.725374,4.565624,1.713907,0.904367,0.240786,2.167092
ENSMUSG00000000031,1.834885,1.096144,11.372461,4.199498,0.814673,1.206164,1.418037,1.716964
ENSMUSG00000000037,1.155056,1.878065,2.099361,0.827695,0.415462,0.016101,0.000000,0.364628
...,...,...,...,...,...,...,...,...
ENSMUSG00000109574,0.414460,0.066366,0.031075,0.021960,0.012043,0.000000,0.000000,0.000000
ENSMUSG00000109575,1.069525,0.000000,0.000000,0.013031,0.000000,0.000000,0.000000,0.000000
ENSMUSG00000109576,0.032109,0.000000,0.000000,0.034781,0.018425,0.000000,0.000000,0.000000


In [45]:
myEuclideanDistLogArr = []
for i in range(0, orthologTableIDs.shape[0]):
    orthologTest = tuple(orthologTableIDs.iloc[i, :])
    humanOrtholog = gtexEPLog[gtexEPLog.index.isin(orthologTest)]
    mouseOrtholog = emtabEPLog[emtabEPLog.index.isin(orthologTest)]

    if not humanOrtholog.empty and not mouseOrtholog.empty:
        myEuclideanDistLogArr.append((orthologTest[0], orthologTest[1], np.linalg.norm(humanOrtholog.iloc[0, :-1] - mouseOrtholog.iloc[0, :-1])))

myEuclidDistLogDF = pd.DataFrame(myEuclideanDistLogArr, columns=["Human ID", "Mouse ID", "EuclidDistLog"])


orthologTableEuclidPearTECNormLog = orthologTableEuclidPearTECNorm.merge(myEuclidDistLogDF, left_on=["Gene stable ID", "Mouse gene stable ID"], right_on=["Human ID", "Mouse ID"], how="left").drop(columns=["Human ID", "Mouse ID"])

In [46]:
myEuclidDistLogDF

,Human ID,Mouse ID,EuclidDistLog
0,ENSG00000198888,ENSMUSG00000064341,6.007114
1,ENSG00000198763,ENSMUSG00000064345,7.318626
2,ENSG00000198804,ENSMUSG00000064351,6.678323
3,ENSG00000198712,ENSMUSG00000064354,40.896612
4,ENSG00000228253,ENSMUSG00000064356,38.358476
...,...,...,...
22008,ENSG00000081692,ENSMUSG00000036819,3.412393
22009,ENSG00000157873,ENSMUSG00000022074,11.676671
22010,ENSG00000157873,ENSMUSG00000042333,9.002332
22011,ENSG00000132676,ENSMUSG00000068921,4.801605


In [47]:
orthologTableEuclidPearTECNorm

,Gene stable ID,Mouse gene stable ID,Mouse homology type,Mouse Gene-order conservation score,EuclidDist,EuclidDistNorm,PearDist,TEC
0,ENSG00000198888,ENSMUSG00000064341,ortholog_one2one,50.0,44492.547302,0.435805,0.209431,0.0000
1,ENSG00000198763,ENSMUSG00000064345,ortholog_one2one,75.0,52572.771576,0.509245,0.297247,0.0000
2,ENSG00000198804,ENSMUSG00000064351,ortholog_one2one,100.0,82619.259132,0.416690,0.235434,0.0000
3,ENSG00000198712,ENSMUSG00000064354,ortholog_one2one,100.0,146743.617287,NaN,NaN,NaN
4,ENSG00000228253,ENSMUSG00000064356,ortholog_one2one,100.0,72933.323800,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
28009,ENSG00000081692,ENSMUSG00000036819,ortholog_one2one,75.0,16.599178,0.668685,0.996874,0.0625
28010,ENSG00000157873,ENSMUSG00000022074,ortholog_one2many,0.0,194.970650,0.543398,0.384453,0.2500
28011,ENSG00000157873,ENSMUSG00000042333,ortholog_one2many,100.0,178.921829,0.834154,0.688115,0.0625
28012,ENSG00000132676,ENSMUSG00000068921,ortholog_one2one,75.0,243.094613,0.516088,0.922319,0.0000


In [48]:
orthologTableEuclidPearTECNormLog.insert(6, "EuclidDistLog", orthologTableEuclidPearTECNormLog.pop("EuclidDistLog"))

In [49]:
orthologTableEuclidPearTECNormLog.to_csv("/Users/andrewhsu/Projects/McNair/data/orthologTableDist.csv", index=False)
orthologTableEuclidPearTECNormLog.to_parquet("/Users/andrewhsu/Projects/McNair/data/orthologTableDist.parquet", index=False)